In [ ]:
import os, psutil
import time
import json
import pickle
import pandas as pd
import numpy as np
from math import ceil

from functools import partial
from itertools import chain
import joblib
from scipy.sparse import save_npz, csr_matrix, vstack


from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [ ]:
import nltk
import networkx as nx
from collections import Counter
from text2graphapi.src.IntegratedSyntacticGraph import ISG

import torch
import optuna
import mlflow
from databricks.sdk import WorkspaceClient

from joblib import Parallel, delayed
import logging

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

Define path variables

In [ ]:
representation_type = "integrated_syntactic_graph"
developer_initials = "JP"

In [ ]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "graph"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"

vocabulary_index_path = current_dir.parent.parent / "data" / "02_models" / "graph" / "vocab_index.pkl"

train_data_isg1_path = current_dir.parent.parent / "data" / "01_processed" / "training-vectors-isg1.npz"
train_data_isg2_path = current_dir.parent.parent / "data" / "01_processed" / "training-vectors-isg2.npz"

val_data_isg1_path = current_dir.parent.parent / "data" / "01_processed" / "val-vectors-isg1.npz"
val_data_isg2_path = current_dir.parent.parent / "data" / "01_processed" / "val-vectors-isg2.npz"

test_data_isg1_path = current_dir.parent.parent / "data" / "01_processed" / "test-vectors-isg1.npz"
test_data_isg2_path = current_dir.parent.parent / "data" / "01_processed" / "test-vectors-isg2.npz"

Connect to databricks for logging results

In [ ]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

What are GPU are the experiments run on

In [ ]:
!nvidia-smi

In [ ]:
running_on_gpu = torch.cuda.is_available()

In [ ]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [ ]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NUMEXPR_MAX_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

Classification threshold constant specification

In [ ]:
classification_thresholds = [x/1000 for x in range(200, 999)]

# Load dataset

#### Load training data

In [ ]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

In [ ]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [ ]:
train_data_df.head(10)

#### Load validation data

In [ ]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

In [ ]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [ ]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

In [ ]:
test_data_df = pd.DataFrame(test_data)

# Functions to build graphs and extract features

In [ ]:
def texts_to_isg_graphs(texts, n_jobs=-1):
    def process(id, text):
        
        logging.disable(logging.INFO)
        logging.getLogger('text2graphapi').setLevel(logging.WARNING)
        logging.getLogger('text2graphapi.models').setLevel(logging.WARNING)
        
        isg = ISG(
            graph_type="DiGraph",
            language="en",
            apply_prep=True,
            output_format="networkx"
        )
        corpus = [{"id": id, "doc": text}]
        graph_object = isg.transform(corpus)[0]["graph"]
        return graph_object

    graphs = Parallel(n_jobs=n_jobs)(
        delayed(process)(id, text)
        for id, text in tqdm(
            enumerate(texts),
            total=len(texts),
            desc="Processing ISGs, print_"
        )
    )

    return graphs

Parse ISG's nodes POS and lemma function

In [ ]:
def parse_graph_node(graph_node):
    node_str = str(graph_node)
    if "_" in node_str:
        lemma, pos = node_str.rsplit("_", 1)
        return lemma.lower(), pos

Parse dependency

In [ ]:
def parse_graph_dependency(data):
    dependency = data.get("gramm_relation")
    parsed_dependency = dependency.split("_", 1)[0]
    return parsed_dependency

Extract multi-level features from graph

In [ ]:
def extract_features_from_isg(graph):
    features = Counter()

    for node in graph.nodes:
        lemma, pos = parse_graph_node(node)
        features[f"LEX::{lemma}"] += 1
        
        if pos:
            features[f"POS::{pos}"] += 1

    for _, _, data in graph.edges(data=True):
        dependency = parse_graph_dependency(data)
        if dependency:
            features[f"DEP::{dependency}"] += 1

    return features

Build vectors based on vocabulary

In [ ]:
def build_vector(features, index):
    vector = np.zeros(len(index), dtype=np.float32)
    for feature, value in features.items():
        if feature in index:
            vector[index[feature]] = value
    return vector

Print process RAM usage

In [ ]:
def print_ram_usage():
    print(f"Process RAM usage: {process.memory_info().rss / 1e9:.2f} GB")

Convert texts to text2graphapi integrated syntactic graphs

In [ ]:
def convert_texts_to_vectors(input_df, index, n_jobs=1, batch_size=3000):
    vectors1 = []
    vectors2 = []
    
    texts1 = input_df["pair"].apply(lambda x: x[0])
    total_batches = ceil(len(texts1) / batch_size)

    for batch_index in tqdm(
    range(total_batches),
    desc="#1 in pair - Processing text batches"
    ):
        start = batch_index * batch_size
        end = start + batch_size
        batch = texts1[start:end]
    
        graphs = texts_to_isg_graphs(batch, n_jobs)
    
        for graph in graphs:
            features = extract_features_from_isg(graph)
            vectors1.append(build_vector(features, index))
    
        del graphs
        gc.collect()
        print_ram_usage()
    
    
    texts2 = input_df["pair"].apply(lambda x: x[1])
    
    for batch_index in tqdm(
        range(total_batches),
        desc="#2 in pair - processing text batches"
    ):
        start = batch_index * batch_size
        end = start + batch_size
        batch = texts2[start:end]
    
        graphs = texts_to_isg_graphs(batch, n_jobs)
    
        for graph in graphs:
            features = extract_features_from_isg(graph)
            vectors2.append(build_vector(features, index))
    
        del graphs
        gc.collect()
        print_ram_usage()
    
    return vectors1, vectors2

Convert a list of texts to vectors

In [ ]:
def convert_texts_list_to_vectors(input_list, index, n_jobs=1, batch_size=3000):
    vectors = []
    
    total_batches = ceil(len(input_list) / batch_size)

    for batch_index in tqdm(
    range(total_batches),
    desc="Processing texts"
    ):
        start = batch_index * batch_size
        end = start + batch_size
        batch = input_list[start:end]
    
        graphs = texts_to_isg_graphs(batch, n_jobs)
    
        for graph in graphs:
            features = extract_features_from_isg(graph)
            vectors.append(csr_matrix(build_vector(features, index)))
    
        del graphs
        gc.collect()
        print_ram_usage()
    
    return vectors

# Build vocabulary index

Build index from the training texts

In [ ]:
def build_index(train_df, n_jobs=1, batch_size=3000):
    index = {}
    next_index_value = 0

    #all training texts
    texts = (
        train_df["pair"].apply(lambda x: x[0]).tolist() +
        train_df["pair"].apply(lambda x: x[1]).tolist()
    )
    
    # total number of batches needed to build the index
    total_batches = ceil(len(texts) / batch_size)

    for batch_index in tqdm(
        range(total_batches),
        desc="Processing batches of training texts"
    ):
        #the first index of a given batch
        start = batch_index * batch_size

        #the last index of a given batch
        end = start + batch_size
        
        #all the texts between the first and last index
        batch = texts[start:end]

        graphs = texts_to_isg_graphs(batch, n_jobs)

        for graph in graphs:
            features =  extract_features_from_isg(graph)
            for feature in features:
                if feature not in index:
                    index[feature] = next_index_value
                    next_index_value += 1

        del graphs
        gc.collect()
        print(len(index))
        print_ram_usage()

    return index

Build index for building vectors first - separately to conserve RAM

process = psutil.Process(os.getpid())
index = build_index(train_data_df, n_jobs=32, batch_size=3000)

len(index)

with open(vocabulary_index_path, "wb") as f:
    pickle.dump(index, f)

# Build the graphs for texts in pair and then vectors out of them and the vocabulary index

In [ ]:
index = None
with open(vocabulary_index_path, "rb") as f:
    index = pickle.load(f)

In [ ]:
process = psutil.Process(os.getpid())

train_texts1 = train_data_df["pair"].apply(lambda x: x[0])
train_vectors1 = convert_texts_list_to_vectors(train_texts1 , index, n_jobs=16, batch_size=3000)

In [ ]:
train_vectors1 = vstack(train_vectors1)
save_npz(train_data_isg1_path, train_vectors1)
del train_vectors1

In [ ]:
process = psutil.Process(os.getpid())

train_texts2 = train_data_df["pair"].apply(lambda x: x[1])
train_vectors2 = convert_texts_list_to_vectors(train_texts2 , index, n_jobs=8, batch_size=3000)

In [ ]:
train_vectors2 = vstack(train_vectors2)
save_npz(train_data_isg2_path, train_vectors2)
del train_vectors2

# Create vectors for testing and validation data and save them

Convert validation data to vectors

val_vectors1, val_vectors2 = convert_texts_to_vectors(val_data_df, index, n_jobs=4, batch_size=3000)

val_vectors1 = csr_matrix(val_vectors1)
val_vectors2 = csr_matrix(val_vectors2)

save_npz(val_data_isg1_path, val_vectors1)
save_npz(val_data_isg2_path, val_vectors2)

del val_vectors1,  val_vectors2

Convert test data to vectors

process = psutil.Process(os.getpid())
test_vectors1, test_vectors2 = convert_texts_to_vectors(test_data_df, index, n_jobs=4, batch_size=3000)

test_vectors1 = csr_matrix(test_vectors1)
test_vectors2 = csr_matrix(test_vectors2)

save_npz(test_data_isg1_path, test_vectors1)
save_npz(test_data_isg2_path, test_vectors2)

del test_vectors1,  test_vectors2